# Analyse par régime de crue (par trajectoire)

Objectif : comparer des modèles (checkpoints) **par trajectoire** et agréger ensuite **par régime** (ex: bins de `Q_peak` extrait du nom du PKL : `Group_X_peak_YYYY_...pkl`).

Sorties :
- un CSV long `per_event_metrics.csv` contenant, pour chaque `exp / epoch / event / horizon` : `MSE(h,u,v)`, `CSI`, et diagnostics wetness (fractions, FP/FN, biais).
- des agrégations par régime (`peak bins`) et des plots.

Notes :
- Le calcul est coûteux : commence avec peu de `DYNAMIC_DIR`, peu d'epochs, et `OVERLAP=1` (échantillonnage grossier) puis densifie.
- On calcule `CSI_all` sur **tous** les nœuds (comme `eval_checkpoints_minimal` / `one_page`) et `CSI_int` sur l'intérieur (BC exclues) pour diagnostic.


In [ ]:
import os, sys, re, math
from pathlib import Path

import torch, dgl  # noqa: F401
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ajouter le repo au PYTHONPATH
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.append(ROOT)

from python.create_dgl_dataset import TelemacDataset
from python.CustomMeshGraphNet import MeshGraphNet
from modulus.launch.utils import load_checkpoint

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
# =====================
# Paramètres utilisateur (à adapter)
# =====================

# Graphe de base (DGL)
DATA_DIR = "/work/m24046/m24046mrcr/paper/Experience2/Multimesh_8_32.bin"

# PKL dynamiques (trajectoires). Idéalement: même liste que tes évaluations globales.
DYNAMIC_DIR = [
    "/work/.../Group_1_peak_2600_Group_1_peak_2600_0_0-80_interpolated.pkl",
    "/work/.../Group_2_peak_1000_Group_2_peak_1000_0_0-80_interpolated.pkl",
]

# Checkpoints à comparer
EXPERIMENTS = {
    # Exemple :
    # "E4_baseline": {"ckpt_dir": "/work/.../Experience4/Seed0/", "epochs": [600, 700, 800, 850, 900]},
    # "E6_tversky": {"ckpt_dir": "/work/.../Experience6/Seed0/", "epochs": [600, 700, 800, 850, 900]},
}

# Paramètres modèle (cohérents avec l'entraînement)
NUM_INPUT_FEATURES = 9
NUM_EDGE_FEATURES = 3
NUM_OUTPUT_FEATURES = 3
MP_LAYERS = 10
DO_CONCAT_TRICK = True
NUM_PROCESSOR_CHECKPOINT_SEGMENTS = 0

# Horizons (pas de 30 min)
HORIZONS_STEPS = [6, 12, 24]
THRESHOLD_M = 0.05

# Séquences évaluées :
# TelemacDataset crée des fenêtres de longueur seq_len avec stride = (seq_len - overlap).
# OVERLAP=1 -> stride ~ seq_len-1 (peu de fenêtres par trajectoire, rapide)
# OVERLAP=seq_len-1 -> stride=1 (toutes les fenêtres, lent)
OVERLAP = 1
MASK_BOUNDARY = True

# Limite pour debug (None = toutes les fenêtres du dataset)
MAX_SEQUENCES_PER_EVENT = None

# Cache
CACHE_CSV = "per_event_metrics.csv"
FORCE_RECOMPUTE = False

# Régimes par bins de Q_peak (extrait du nom)
PEAK_BINS = [0, 1600, 2400, 3200, float("inf")]
PEAK_LABELS = ["<1600", "1600-2400", "2400-3200", ">=3200"]


In [ ]:
# =====================
# Parsing des trajectoires (Group / peak)
# =====================

_RE = re.compile(r"Group_(?P<group>\d+)_peak_(?P<peak>\d+)")

def parse_event_info(path: str):
    base = os.path.basename(path)
    m = _RE.search(base)
    if m is None:
        return {
            "event": base,
            "group": None,
            "peak": None,
        }
    return {
        "event": base,
        "group": int(m.group("group")),
        "peak": int(m.group("peak")),
    }

events_df = pd.DataFrame([parse_event_info(p) | {"path": p} for p in DYNAMIC_DIR])
events_df

In [ ]:
# =====================
# Modèle / dataset helpers
# =====================

def build_model():
    return MeshGraphNet(
        NUM_INPUT_FEATURES,
        NUM_EDGE_FEATURES,
        NUM_OUTPUT_FEATURES,
        processor_size=MP_LAYERS,
        hidden_dim_processor=64,
        hidden_dim_node_encoder=64,
        hidden_dim_edge_encoder=64,
        hidden_dim_node_decoder=64,
        do_concat_trick=DO_CONCAT_TRICK,
        num_processor_checkpoint_segments=NUM_PROCESSOR_CHECKPOINT_SEGMENTS,
    )

def load_model_checkpoint(model, ckpt_dir, epoch):
    load_checkpoint(ckpt_dir, models=model, device=device, epoch=epoch)
    model.to(device)
    model.eval()
    return model

def build_dataset_for_event(dynamic_file, ckpt_dir, sequence_length, overlap, split="test"):
    return TelemacDataset(
        name=f"eval_{split}",
        data_dir=DATA_DIR,
        dynamic_data_files=[dynamic_file],
        split=split,
        ckpt_path=ckpt_dir,
        normalize=True,
        sequence_length=sequence_length,
        overlap=overlap,
    )

def _denorm(xn, mean, std):
    return xn * std + mean

def _renorm(x, mean, std):
    return (x - mean) / (std + 1e-12)


In [ ]:
# =====================
# Évaluation : métriques par event
# =====================

def evaluate_model_per_event(
    model,
    ds,
    horizons_steps,
    threshold=0.05,
    mask_boundary=True,
    max_sequences=None,
):
    stats = ds.node_stats
    dyn_start = ds.base_graph.ndata["static"].shape[1]

    mx = torch.tensor([stats["h"].item(), stats["u"].item(), stats["v"].item()], device=device)
    sx = torch.tensor([stats["h_std"].item(), stats["u_std"].item(), stats["v_std"].item()], device=device)
    dy_mean = torch.tensor([stats["delta_h"].item(), stats["delta_u"].item(), stats["delta_v"].item()], device=device)
    dy_std = torch.tensor([
        stats["delta_h_std"].item(),
        stats["delta_u_std"].item(),
        stats["delta_v_std"].item(),
    ], device=device)

    agg = {
        h: {
            # MSE sur tous les nœuds (BC incluses)
            "count_all": 0,
            "sqerr_sum_all": np.zeros(3, dtype=np.float64),
            "tp_all": 0,
            "fp_all": 0,
            "fn_all": 0,
            # Wetness diagnostics sur intérieur (BC exclues)
            "count_int": 0,
            "bias_sum_int": 0.0,
            "wet_pred_sum_int": 0,
            "wet_gt_sum_int": 0,
            "tp_int": 0,
            "fp_int": 0,
            "fn_int": 0,
        }
        for h in horizons_steps
    }

    nseq = len(ds) if max_sequences is None else min(max_sequences, len(ds))
    max_h = max(horizons_steps)

    for idx in range(nseq):
        graphs = ds[idx]
        if len(graphs) <= max_h:
            continue

        g = graphs[0].to(device)
        static_part = g.ndata["x"][:, :dyn_start]
        xn_t = g.ndata["x"][:, dyn_start : dyn_start + 3]

        onehot = static_part[:, :4]
        q_mask = (onehot == torch.tensor([0, 0, 1, 0], device=device)).all(dim=1)
        h_mask = (onehot == torch.tensor([0, 1, 0, 0], device=device)).all(dim=1)
        interior_mask = ~(q_mask | h_mask) if mask_boundary else torch.ones_like(q_mask)

        for t in range(max_h):
            with torch.no_grad():
                y_pred_n = model(g.ndata["x"], g.edata["x"], g)

            x_t = _denorm(xn_t, mx, sx)
            y_pred = _denorm(y_pred_n, dy_mean, dy_std)
            x_t1 = x_t + y_pred

            x_gt_n = graphs[t + 1].ndata["x"][:, dyn_start : dyn_start + 3].to(device)
            x_gt = _denorm(x_gt_n, mx, sx)

            # BC injection
            x_t1[q_mask] = x_gt[q_mask]
            x_t1[h_mask, 0:1] = x_gt[h_mask, 0:1]

            step = t + 1
            if step in horizons_steps:
                a = agg[step]

                # --- MSE all nodes ---
                diff_all = x_t1 - x_gt
                n_all = int(diff_all.shape[0])
                a["count_all"] += n_all
                a["sqerr_sum_all"] += (diff_all**2).sum(dim=0).detach().cpu().numpy()

                # --- CSI all nodes ---
                h_pred_all = x_t1[:, 0].detach().cpu().numpy()
                h_gt_all = x_gt[:, 0].detach().cpu().numpy()
                pred_wet_all = h_pred_all >= threshold
                gt_wet_all = h_gt_all >= threshold
                a["tp_all"] += int(np.logical_and(pred_wet_all, gt_wet_all).sum())
                a["fp_all"] += int(np.logical_and(pred_wet_all, ~gt_wet_all).sum())
                a["fn_all"] += int(np.logical_and(~pred_wet_all, gt_wet_all).sum())

                # --- Wetness diagnostics interior ---
                h_pred = x_t1[:, 0][interior_mask]
                h_gt = x_gt[:, 0][interior_mask]
                if h_pred.numel() > 0:
                    hp = h_pred.detach().cpu().numpy()
                    hg = h_gt.detach().cpu().numpy()
                    pred_wet = hp >= threshold
                    gt_wet = hg >= threshold
                    n = int(hp.size)
                    a["count_int"] += n
                    a["bias_sum_int"] += float((hp - hg).sum())
                    a["wet_pred_sum_int"] += int(pred_wet.sum())
                    a["wet_gt_sum_int"] += int(gt_wet.sum())
                    a["tp_int"] += int(np.logical_and(pred_wet, gt_wet).sum())
                    a["fp_int"] += int(np.logical_and(pred_wet, ~gt_wet).sum())
                    a["fn_int"] += int(np.logical_and(~pred_wet, gt_wet).sum())

            # reinjection
            xn_t = _renorm(x_t1, mx, sx)
            g = g.clone()
            g.ndata["x"] = torch.cat([static_part, xn_t], dim=1)

    summary = {}
    for h in horizons_steps:
        a = agg[h]

        # MSE (all)
        if a["count_all"] > 0:
            mse = (a["sqerr_sum_all"] / a["count_all"]).tolist()
        else:
            mse = [math.nan, math.nan, math.nan]

        # CSI (all)
        denom_all = a["tp_all"] + a["fp_all"] + a["fn_all"]
        csi_all = (a["tp_all"] / denom_all) if denom_all > 0 else math.nan

        # Wetness diagnostics (interior)
        if a["count_int"] > 0:
            wet_pred = a["wet_pred_sum_int"] / a["count_int"]
            wet_gt = a["wet_gt_sum_int"] / a["count_int"]
            denom_int = a["tp_int"] + a["fp_int"] + a["fn_int"]
            csi_int = (a["tp_int"] / denom_int) if denom_int > 0 else math.nan
            bias_mean = a["bias_sum_int"] / a["count_int"]
            fp_rate = a["fp_int"] / a["count_int"]
            fn_rate = a["fn_int"] / a["count_int"]
        else:
            wet_pred = math.nan
            wet_gt = math.nan
            csi_int = math.nan
            bias_mean = math.nan
            fp_rate = math.nan
            fn_rate = math.nan

        summary[h] = {
            "mse_h": float(mse[0]),
            "mse_u": float(mse[1]),
            "mse_v": float(mse[2]),
            "mse_mean": float(np.nanmean(mse)),
            "csi_all": float(csi_all),
            "csi_int": float(csi_int),
            "bias_mean": float(bias_mean),
            "wet_frac_pred": float(wet_pred),
            "wet_frac_gt": float(wet_gt),
            "wet_frac_delta": float(wet_pred - wet_gt) if (wet_pred == wet_pred and wet_gt == wet_gt) else math.nan,
            "false_pos_rate": float(fp_rate),
            "false_neg_rate": float(fn_rate),
        }
    return summary


def build_event_datasets(dynamic_files, ckpt_dir, sequence_length, overlap):
    ds_map = {}
    for p in dynamic_files:
        ds_map[p] = build_dataset_for_event(p, ckpt_dir, sequence_length, overlap, split="test")
    return ds_map


def run_regime_eval(experiments):
    rows = []
    seq_len = max(HORIZONS_STEPS) + 1

    for exp_name, cfg in experiments.items():
        ckpt_dir = cfg["ckpt_dir"]
        epochs = cfg["epochs"]

        # cache datasets (normalisés avec les stats de cet exp)
        ds_map = build_event_datasets(DYNAMIC_DIR, ckpt_dir, sequence_length=seq_len, overlap=OVERLAP)

        for ep in epochs:
            model = build_model()
            load_model_checkpoint(model, ckpt_dir, epoch=ep)

            for event_path, ds in ds_map.items():
                info = parse_event_info(event_path)
                summ = evaluate_model_per_event(
                    model,
                    ds,
                    horizons_steps=HORIZONS_STEPS,
                    threshold=THRESHOLD_M,
                    mask_boundary=MASK_BOUNDARY,
                    max_sequences=MAX_SEQUENCES_PER_EVENT,
                )
                for h, vals in summ.items():
                    rows.append({
                        "exp": exp_name,
                        "epoch": ep,
                        "horizon_steps": h,
                        **info,
                        **vals,
                    })

    return pd.DataFrame(rows)


In [ ]:
# =====================
# Run (ou recharge cache)
# =====================

cache_path = Path(CACHE_CSV)
if cache_path.exists() and not FORCE_RECOMPUTE:
    df = pd.read_csv(cache_path)
else:
    df = run_regime_eval(EXPERIMENTS)
    df.to_csv(cache_path, index=False)

df.head()

In [ ]:
# =====================
# Agrégation par régime (bins de peak)
# =====================

df = df.copy()
df["regime"] = pd.cut(df["peak"], bins=PEAK_BINS, labels=PEAK_LABELS, right=False)

group_cols = ["exp", "epoch", "horizon_steps", "regime"]
agg = (
    df.groupby(group_cols)
    .agg(
        n_events=("event", "nunique"),
        csi_all_mean=("csi_all", "mean"),
        csi_int_mean=("csi_int", "mean"),
        mse_h_mean=("mse_h", "mean"),
        mse_mean_mean=("mse_mean", "mean"),
        wet_pred_mean=("wet_frac_pred", "mean"),
        wet_gt_mean=("wet_frac_gt", "mean"),
        fp_rate_mean=("false_pos_rate", "mean"),
        fn_rate_mean=("false_neg_rate", "mean"),
        bias_mean=("bias_mean", "mean"),
    )
    .reset_index()
)

agg.sort_values(["horizon_steps", "epoch", "exp", "regime"]).head(20)

In [ ]:
# =====================
# (Optionnel) Agrégation par "Group" (forme d'hydrogramme)
# =====================

group_cols_g = ["exp", "epoch", "horizon_steps", "group"]
agg_group = (
    df.groupby(group_cols_g)
    .agg(
        n_events=("event", "nunique"),
        csi_all_mean=("csi_all", "mean"),
        csi_int_mean=("csi_int", "mean"),
        mse_h_mean=("mse_h", "mean"),
        mse_mean_mean=("mse_mean", "mean"),
        wet_pred_mean=("wet_frac_pred", "mean"),
        wet_gt_mean=("wet_frac_gt", "mean"),
        fp_rate_mean=("false_pos_rate", "mean"),
        fn_rate_mean=("false_neg_rate", "mean"),
        bias_mean=("bias_mean", "mean"),
    )
    .reset_index()
)

agg_group.sort_values(["horizon_steps", "epoch", "exp", "group"]).head(20)

In [ ]:
# =====================
# Plots rapides
# =====================

def plot_csi_by_regime(agg_df, horizon_steps, metric="csi_all_mean"):
    sub = agg_df[agg_df["horizon_steps"].eq(horizon_steps)].copy()
    if sub.empty:
        print("No data")
        return

    exps = sorted(sub["exp"].unique())
    regimes = [r for r in PEAK_LABELS if r in sub["regime"].astype(str).unique()]

    fig, ax = plt.subplots(figsize=(10, 3.5))
    for exp in exps:
        g = sub[sub["exp"].eq(exp)].sort_values(["epoch", "regime"])
        # pivot: index=epoch, columns=regime
        pv = g.pivot_table(index="epoch", columns="regime", values=metric)
        for reg in regimes:
            if reg not in pv.columns:
                continue
            ax.plot(pv.index, pv[reg], marker="o", label=f"{exp} | {reg}")

    ax.set_title(f"{metric} by peak regime — horizon={horizon_steps}")
    ax.set_xlabel("epoch")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8, ncol=2)
    plt.tight_layout();
    plt.show()

for h in HORIZONS_STEPS:
    plot_csi_by_regime(agg, horizon_steps=h, metric="csi_all_mean")
